<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/embeddings/hobbylm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HobbyLM-Embed

[HobbyLM-Embed-140M](https://huggingface.co/harims95/hobbylm-embed-140m) is an English text-embedding model with a sparse mixture-of-experts encoder: 140M total parameters, 36M active per token (excluding embeddings), 512-dimensional output, Apache-2.0. This notebook uses it through the existing `HuggingFaceEmbedding` class.

- `trust_remote_code=True` is required: the encoder is custom code in the model repository. The examples pin `revision` to a tested version of that code.
- For retrieval, the model expects the prefix `query: ` on queries and `passage: ` on documents. Pass them as `query_instruction` and `text_instruction`; `HuggingFaceEmbedding` then adds them for you, so pass raw text.

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index-embeddings-huggingface

## Load the model

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(
    model_name="harims95/hobbylm-embed-140m",
    revision="ea708cee107818fa0c1871eebd26d82a1e3fb1aa",
    trust_remote_code=True,
    query_instruction="query: ",
    text_instruction="passage: ",
)

## Embed queries and documents

The vectors have 512 dimensions and unit length.

In [ ]:
queries = ["how do vaccines work?", "what is the capital of france"]
passages = [
    "Vaccines train the immune system to recognise a pathogen without causing the disease.",
    "Paris is the capital and largest city of France.",
]

q = [embed_model.get_query_embedding(t) for t in queries]
p = embed_model.get_text_embedding_batch(passages)
print(
    "dims",
    len(q[0]),
    "norms",
    [round(sum(x * x for x in v) ** 0.5, 4) for v in q + p],
)

dims 512 norms [1.0, 1.0, 1.0, 1.0]


## Retrieve with a `VectorStoreIndex`

With `Settings.embed_model` set, documents are embedded with the `passage: ` prefix and queries with the `query: ` prefix.

In [ ]:
from llama_index.core import Document, Settings, VectorStoreIndex

Settings.embed_model = embed_model
index = VectorStoreIndex.from_documents([Document(text=t) for t in passages])
retriever = index.as_retriever(similarity_top_k=1)
for t in queries:
    print(t, "->", retriever.retrieve(t)[0].node.get_content())

how do vaccines work? -> Vaccines train the immune system to recognise a pathogen without causing the disease.
what is the capital of france -> Paris is the capital and largest city of France.


## Long documents

The model reads 512 tokens by default. For long documents, `max_length=2048` lets it read up to 2,048 tokens; longer inputs are truncated.

In [ ]:
long_embed_model = HuggingFaceEmbedding(
    model_name="harims95/hobbylm-embed-140m",
    revision="ea708cee107818fa0c1871eebd26d82a1e3fb1aa",
    trust_remote_code=True,
    query_instruction="query: ",
    text_instruction="passage: ",
    max_length=2048,
)

long_text = " ".join(passages * 100)
vector = long_embed_model.get_text_embedding(long_text)
print(long_embed_model.max_length, len(vector))

2048 512


For classification, clustering or similarity tasks, encode plain text without a prefix (for example with Sentence Transformers' `model.encode(texts)`).